# 01 — Platform and methodology

**Question.** Under what conditions were the numbers taken, and how were they validated?

Everything in the analysis derives from raw per-process records kept unmodified in `data/raw/`. This notebook documents the platform, how each campaign was launched, the validity rules, and how repeatable the measurements are.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench import report, roofline, plotting
from hwbench.notebook import setup
ctx = setup('01_platform_and_methodology')
thesis, ext, theory = ctx.thesis, ctx.ext, ctx.theory
table, figure, say, say_claims = ctx.table, ctx.figure, ctx.say, ctx.say_claims
from hwbench import claims

## 1. Platform record

Read on the execution node, without root, for one Flat and one Cache campaign. An unreadable source is recorded as such: for example, RAPL and MSR access is restricted, which bounds which counters can be used.

In [ ]:
import json
def platform_of(mode):
    camp = sorted(ext[ext.memory_mode == mode].campaign_id.unique())[0]
    return camp, json.loads((ROOT / 'data/raw' / mode / camp / 'system/platform.json').read_text()), json.loads((ROOT / 'data/raw' / mode / camp / 'system/topology.json').read_text())
rows = []
for mode in ['flat', 'cache']:
    camp, plat, topo = platform_of(mode)
    proc, pol = plat['processor'], plat['kernel_policy']
    rows.append({
        'mode': mode, 'campaign': camp, 'node': topo['hostname'].split('.')[0], 'cpu': topo['cpu_model'],
        'sockets': topo['socket_count'], 'cores': topo['physical_cores'], 'threads/core': topo['threads_per_core'],
        'NUMA nodes': len(topo['numa_nodes']), 'kernel': plat['os']['kernel'], 'OS': plat['os']['pretty_name'],
        'cpufreq max (MHz)': proc['cpuinfo_max_khz'] / 1000, 'THP': pol.get('transparent_hugepage'),
        'perf_event_paranoid': pol.get('perf_event_paranoid'), 'NUMA balancing': pol.get('numa_balancing'),
    })
table(pd.DataFrame(rows).set_index('mode').T.reset_index().rename(columns={'index': 'property'}))

In [ ]:
camp, plat, topo = platform_of('flat')
hm = plat['hmat_access']
nodes = {n['node']: n for n in topo['numa_nodes']}
table(pd.DataFrame([dict(node=int(k), tier=nodes[int(k)]['memory_tier'], socket=nodes[int(k)]['socket'], gib=round(nodes[int(k)]['memory_gib'], 1),
        hmat_read_GBs=v['read_bandwidth'] / 1000, hmat_write_GBs=v['write_bandwidth'] / 1000,
        hmat_read_ns=v['read_latency']) for k, v in hm.items()]))

In [ ]:
say_claims(claims.core_scaling(ext, thesis, ROOT) + claims.latency(ext, theory, ROOT), ['hmat_gap', 'hmat_latency'])

**Platform caveats.** Four settings of the execution nodes shape what the numbers mean. They were read from every campaign's platform record and are identical in all of them.

In [ ]:
say_claims(claims.platform_caveats(ext, ROOT))

## 2. Execution rules and provenance

* **Flat** campaigns run on the single dedicated Flat node, outside Slurm, in a tmux session started over SSH. Before measuring, the runner checks the node is idle (no Slurm job placed on it, 1-minute load below 1), and every record is bracketed by load and job snapshots. A foreign job observed at any point marks the record `interference_suspected` and processing rejects it.
* **Cache** campaigns run only under Slurm, exclusively (`--exclusive --mem=0`), with the Flat node excluded in the versioned submission template; the runner aborts if the node is the Flat node or its topology is not Cache.
* The October-11 Flat campaigns were run through an exclusive Slurm allocation on the same node (`launch_method = slurm_exclusive`); they are reused unchanged, and the cross-check below compares them with SSH runs.

In [ ]:
sel = pd.concat([thesis, ext])
prov = (sel.groupby(['memory_mode', 'campaign_id', 'launch_method', 'node', 'memory_mode_verified'], dropna=False)
        .agg(samples=('accepted', 'size'), source_sha256=('source_sha256', lambda s: ';'.join(sorted({x[:8] for x in s.dropna()})))).reset_index())
table(prov)

## 3. Validity rules and repeatability

A sample is used only if the benchmark's own checksum validated, the process ran on the expected host with the expected topology, its clock window lies inside the process window, the disassembly of the compiled benchmark has the intended instruction in an innermost loop with as many copies as the kernel has chains or tiles (`hwbench.disasm`: loops are recovered from backward branches in the retained `objdump`; it shows the loop exists and what it holds, not that the timed call reaches it), GEMMs show an AMX execution in oneDNN's verbose log, and no foreign Slurm job was seen. Each configuration needs five steady samples; variation above 5% is flagged, not hidden.

In [ ]:
v = ctx.validation
say(f"**Validation.** status `{v['status']}`; {v['selected_samples']} thesis samples, {v['extended_samples']} extended samples; "
    f"{len(v['errors'])} errors, {len(v['warnings'])} warnings.")
table(pd.DataFrame({'warning': v['warnings']}))

In [ ]:
cvt = report.sample_cv(ext)
summary = cvt.groupby('family').cv_percent.describe(percentiles=[.5, .9, .99])[['count', '50%', '90%', '99%', 'max']]
summary.loc['all'] = cvt.cv_percent.describe(percentiles=[.5, .9, .99])[['count', '50%', '90%', '99%', 'max']]
table(summary.rename(columns={'count': 'configurations', '50%': 'median CV %', '90%': '90th %', '99%': '99th %', 'max': 'max %'}).reset_index(), 1)

The table groups samples by *configuration*: benchmark, tier, thread count, working set and, for GEMM, shape and batch. The ten largest spreads are listed rather than hidden:

In [ ]:
worst = cvt.sort_values('cv_percent', ascending=False).head(10)
table(worst[['memory_mode', 'benchmark', 'dtype', 'memory_target', 'm', 'n', 'k', 'cv_percent', 'n_samples']], 1)
say_claims(claims.repeatability(ext, thesis), ['per_configuration_cv'])

The first of the five windows of a configuration can differ from the rest. The table compares it with the median of the other four:

In [ ]:
fw = report.first_window_effect(ext)
fw['pool'] = np.where(fw.working_set_bytes > 64 * 2**30, 'spilled (>64 GiB)', 'other')
# Bandwidth is split by tier and by 56 cores against fewer: the HBM transient only shows at 56 cores.
fw['tier'] = np.where(fw.family.eq('bandwidth'), fw.memory_target.fillna(''), '')
fw['cores'] = np.where(fw.family.eq('bandwidth'), np.where(fw.threads.eq(56), '56', 'fewer'), '')
view = fw.groupby(['memory_mode', 'family', 'tier', 'cores', 'pool']).agg(configurations=('ratio', 'size'), median_ratio=('ratio', 'median'),
        lowest_ratio=('ratio', 'min'), highest_ratio=('ratio', 'max'), below_90_percent=('ratio', lambda r: (r < 0.9).mean()),
        above_103_percent=('ratio', lambda r: (r > 1.03).mean()), median_shift=('median_shift', lambda r: r.abs().max())).reset_index()
table(view[(view.configurations > 0)], 3)
say_claims(claims.repeatability(ext, thesis), ['cold_first_window', 'hbm_first_window'])

## 4. Repeat run against the October values

Flat repeats run over SSH on the same node; Cache repeats run on a second node. The acceptance threshold is 2%. Whether a Cache deviation is a property of the node is tested below against the spread between campaigns on one node.

In [ ]:
x = report.xcheck_table(ext, thesis)
table(x[['memory_mode', 'benchmark', 'dtype', 'memory_target', 'socket', 'node', 'new', 'october', 'october_launch', 'deviation_percent', 'within_tolerance', 'cv_percent']])

In [ ]:
flat_x, cache_x = x[x.memory_mode == 'flat'], x[x.memory_mode == 'cache']
say(f"**Result.** Flat, SSH versus the October Slurm-exclusive runs: {flat_x.within_tolerance.mean():.0%} of {len(flat_x)} points within 2% "
    f"(largest {flat_x.deviation_percent.abs().max():.1f}%). Cache, second node versus the first: {cache_x.within_tolerance.mean():.0%} of {len(cache_x)} within 2% "
    f"(largest {cache_x.deviation_percent.abs().max():.1f}%).")

In [ ]:
table(report.repeat_ranges(ext).rename(columns={'min': 'lowest launch', 'max': 'highest launch', 'mean': 'mean of launches'}), 3)
say_claims(claims.xcheck(ext, thesis), ['repeat_runs', 'repeat_launches'])

**Between campaigns.** Every selected campaign that measured a headline configuration (socket 0, 56 cores, 8 GiB, and 96 GiB in Cache mode) gives an independent launch on another day. *Same node* is the mode's reference node (hbm14 for Flat, hbm01 for Cache):

In [ ]:
cs = report.campaign_spread(ext, thesis)
table(cs[['memory_mode', 'quantity', 'working_set_gib', 'campaigns', 'same_node_campaigns', 'same_node_low', 'same_node_high', 'same_node_percent', 'all_nodes_percent', 'nodes']], 2)
say_claims(claims.xcheck(ext, thesis), ['campaign_spread'])

The same launches as a figure: each dot is the median of one launch relative to the mean of the three, the line joins the lowest and the highest, and the band is the 2% acceptance threshold used for the cross-check above. A row is one quantity in one memory mode; the three launches of each ran on one node.

In [ ]:
fig = plotting.repeat_runs_figure(report.repeat_launches(ext))
figure(fig, 'repeat_runs')

## 5. Traffic and operation conventions

* Triad counts 24 B per element (two reads, one write); read-only counts 8 B per element; operations are counted as multiply + add = 2.
* GEMM traffic is weights + input activations + output per call, with weights drawn round-robin from a pool larger than four times the last-level cache so they stream from memory.
* Effective clock is user-mode core cycles ÷ (threads × wall time), self-monitored per thread through `perf_event_open`; AMX-busy is the `EXE.AMX_BUSY` event over the same cycles.